## Sending Test Event to Event hub

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json


EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="key-vault-scope",
    key="eventhub-connection-string"
)
EVENT_HUB_NAME = "pmb-weather-streaming-eventhub"

# Initialize producer
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

def send_event(event):
    # Create batch
    event_data_batch = producer.create_batch()

    # Add event
    event_data_batch.add(EventData(json.dumps(event)))

    # Send batch
    producer.send_batch(event_data_batch)


# Sample JSON event
event = {
    "event_id": 2222,
    "event_name": "key vault Test Event"
}

# Send event
send_event(event)

# Close producer
producer.close()

## API Testing

In [0]:
import requests
import json

# Get secret value from Azure Key Vault
weather_api_key = dbutils.secrets.get(
    scope="key-vault-scope",
    key="apikey"
)

# Location
location = "Chennai"

# Base URL
base_url = "http://api.weatherapi.com/v1/current.json"

# Params
params = {
    "key": weather_api_key,
    "q": location
}

# API call
response = requests.get(base_url, params=params)

# Response handling
if response.status_code == 200:
    current_weather = response.json()
    print("Current Weather:")
    print(json.dumps(current_weather, indent=3))
else:
    print(f"Error: {response.status_code}, {response.text}")

Current Weather:
{
   "location": {
      "name": "Chennai",
      "region": "Tamil Nadu",
      "country": "India",
      "lat": 13.0833,
      "lon": 80.2833,
      "tz_id": "Asia/Kolkata",
      "localtime_epoch": 1774613294,
      "localtime": "2026-03-27 17:38"
   },
   "current": {
      "last_updated_epoch": 1774612800,
      "last_updated": "2026-03-27 17:30",
      "temp_c": 31.2,
      "temp_f": 88.2,
      "is_day": 1,
      "condition": {
         "text": "Sunny",
         "icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
         "code": 1000
      },
      "wind_mph": 14.8,
      "wind_kph": 23.8,
      "wind_degree": 152,
      "wind_dir": "SSE",
      "pressure_mb": 1007.0,
      "pressure_in": 29.74,
      "precip_mm": 0.0,
      "precip_in": 0.0,
      "humidity": 71,
      "cloud": 25,
      "feelslike_c": 35.8,
      "feelslike_f": 96.3,
      "windchill_c": 28.9,
      "windchill_f": 84.1,
      "heatindex_c": 31.4,
      "heatindex_f": 88.4,
      "dewpoin

## Complete code for getting Weather data

In [0]:
import requests
import json

# -------------------------------
# Handle API Response
# -------------------------------
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return {
            "error": f"{response.status_code}: {response.text}"
        }

# -------------------------------
# Get Current Weather
# -------------------------------
def get_current_weather(base_url, api_key, location):
    url = f"{base_url}/current.json"
    params = {
        "key": api_key,
        "q": location,
        "aqi": "yes"
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Get Forecast
# -------------------------------
def get_forecast_weather(base_url, api_key, location, days):
    url = f"{base_url}/forecast.json"
    params = {
        "key": api_key,
        "q": location,
        "days": days
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Get Alerts
# -------------------------------
def get_alerts(base_url, api_key, location):
    url = f"{base_url}/alerts.json"
    params = {
        "key": api_key,
        "q": location,
        "alerts": "yes"
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Flatten Data
# -------------------------------
def flatten_data(current_weather, forecast_weather, alerts):

    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})

    forecast_days = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        "name": location_data.get("name"),
        "region": location_data.get("region"),
        "country": location_data.get("country"),
        "lat": location_data.get("lat"),
        "lon": location_data.get("lon"),
        "localtime": location_data.get("localtime"),

        "temp_c": current.get("temp_c"),
        "is_day": current.get("is_day"),

        "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"),

        "wind_kph": current.get("wind_kph"),
        "wind_degree": current.get("wind_degree"),

        "air_quality": {
            "co": air_quality.get("co"),
            "no2": air_quality.get("no2"),
            "o3": air_quality.get("o3"),
            "so2": air_quality.get("so2"),
            "pm2_5": air_quality.get("pm2_5"),
            "pm10": air_quality.get("pm10"),
            "us_epa_index": air_quality.get("us-epa-index"),
            "gb_defra_index": air_quality.get("gb-defra-index")
        },

        "alerts": [
            {
                "headline": alert.get("headline"),
                "severity": alert.get("severity"),
                "description": alert.get("desc"),
                "instruction": alert.get("instruction")
            }
            for alert in alert_list
        ],

        "forecast": [
            {
                "date": day.get("date"),
                "maxtemp_c": day.get("day", {}).get("maxtemp_c"),
                "mintemp_c": day.get("day", {}).get("mintemp_c"),
                "condition": day.get("day", {}).get("condition", {}).get("text")
            }
            for day in forecast_days
        ]
    }

    return flattened_data

# -------------------------------
# Main Function
# -------------------------------
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1"
    location = "Chennai"

    api_key = dbutils.secrets.get(
        scope="key-vault-scope",
        key="apikey"
    )

    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    alerts = get_alerts(base_url, api_key, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)

    print(json.dumps(merged_data, indent=3))


# Run
fetch_weather_data()

{
   "name": "Chennai",
   "region": "Tamil Nadu",
   "country": "India",
   "lat": 13.0833,
   "lon": 80.2833,
   "localtime": "2026-03-27 18:04",
   "temp_c": 30.1,
   "is_day": 1,
   "condition_text": "Clear",
   "condition_icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
   "wind_kph": 22.3,
   "wind_degree": 153,
   "air_quality": {
      "co": 249.85,
      "no2": 4.45,
      "o3": 113.0,
      "so2": 12.05,
      "pm2_5": 21.55,
      "pm10": 36.35,
      "us_epa_index": 2,
      "gb_defra_index": 2
   },
   "alerts": [],
   "forecast": [
      {
         "date": "2026-03-27",
         "maxtemp_c": 30.6,
         "mintemp_c": 24.4,
         "condition": "Sunny"
      },
      {
         "date": "2026-03-28",
         "maxtemp_c": 31.6,
         "mintemp_c": 24.9,
         "condition": "Sunny"
      },
      {
         "date": "2026-03-29",
         "maxtemp_c": 32.5,
         "mintemp_c": 26.3,
         "condition": "Sunny"
      }
   ]
}


## Sending Complete Data to Event Hub

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json

import requests

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="key-vault-scope",
    key="eventhub-connection-string"
)
EVENT_HUB_NAME = "pmb-weather-streaming-eventhub"

# Initialize producer
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)


def send_event(event):
    # Create batch
    event_data_batch = producer.create_batch()

    # Add event
    event_data_batch.add(EventData(json.dumps(event)))

    # Send batch
    producer.send_batch(event_data_batch)


# -------------------------------
# Handle API Response
# -------------------------------
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return {
            "error": f"{response.status_code}: {response.text}"
        }

# -------------------------------
# Get Current Weather
# -------------------------------
def get_current_weather(base_url, api_key, location):
    url = f"{base_url}/current.json"
    params = {
        "key": api_key,
        "q": location,
        "aqi": "yes"
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Get Forecast
# -------------------------------
def get_forecast_weather(base_url, api_key, location, days):
    url = f"{base_url}/forecast.json"
    params = {
        "key": api_key,
        "q": location,
        "days": days
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Get Alerts
# -------------------------------
def get_alerts(base_url, api_key, location):
    url = f"{base_url}/alerts.json"
    params = {
        "key": api_key,
        "q": location,
        "alerts": "yes"
    }

    response = requests.get(url, params=params)
    return handle_response(response)

# -------------------------------
# Flatten Data
# -------------------------------
def flatten_data(current_weather, forecast_weather, alerts):

    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})

    forecast_days = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        "name": location_data.get("name"),
        "region": location_data.get("region"),
        "country": location_data.get("country"),
        "lat": location_data.get("lat"),
        "lon": location_data.get("lon"),
        "localtime": location_data.get("localtime"),

        "temp_c": current.get("temp_c"),
        "is_day": current.get("is_day"),

        "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"),

        "wind_kph": current.get("wind_kph"),
        "wind_degree": current.get("wind_degree"),

        "air_quality": {
            "co": air_quality.get("co"),
            "no2": air_quality.get("no2"),
            "o3": air_quality.get("o3"),
            "so2": air_quality.get("so2"),
            "pm2_5": air_quality.get("pm2_5"),
            "pm10": air_quality.get("pm10"),
            "us_epa_index": air_quality.get("us-epa-index"),
            "gb_defra_index": air_quality.get("gb-defra-index")
        },

        "alerts": [
            {
                "headline": alert.get("headline"),
                "severity": alert.get("severity"),
                "description": alert.get("desc"),
                "instruction": alert.get("instruction")
            }
            for alert in alert_list
        ],

        "forecast": [
            {
                "date": day.get("date"),
                "maxtemp_c": day.get("day", {}).get("maxtemp_c"),
                "mintemp_c": day.get("day", {}).get("mintemp_c"),
                "condition": day.get("day", {}).get("condition", {}).get("text")
            }
            for day in forecast_days
        ]
    }

    return flattened_data

# -------------------------------
# Main Function
# -------------------------------
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1"
    location = "Chennai"

    api_key = dbutils.secrets.get(
        scope="key-vault-scope",
        key="apikey"
    )

    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    alerts = get_alerts(base_url, api_key, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)

    print(json.dumps(merged_data, indent=3))

    # Sending Event Data
    send_event(merged_data)
    
# Run
fetch_weather_data()

{
   "name": "Chennai",
   "region": "Tamil Nadu",
   "country": "India",
   "lat": 13.0833,
   "lon": 80.2833,
   "localtime": "2026-03-27 18:14",
   "temp_c": 30.2,
   "is_day": 1,
   "condition_text": "Clear",
   "condition_icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
   "wind_kph": 22.3,
   "wind_degree": 153,
   "air_quality": {
      "co": 249.85,
      "no2": 4.45,
      "o3": 113.0,
      "so2": 12.05,
      "pm2_5": 21.55,
      "pm10": 36.35,
      "us_epa_index": 2,
      "gb_defra_index": 2
   },
   "alerts": [],
   "forecast": [
      {
         "date": "2026-03-27",
         "maxtemp_c": 30.6,
         "mintemp_c": 24.4,
         "condition": "Sunny"
      },
      {
         "date": "2026-03-28",
         "maxtemp_c": 31.6,
         "mintemp_c": 24.9,
         "condition": "Sunny"
      },
      {
         "date": "2026-03-29",
         "maxtemp_c": 32.5,
         "mintemp_c": 26.3,
         "condition": "Sunny"
      }
   ]
}


## Sending Weather Data in Streaming Fashion

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json
import requests
import time

# -------------------------------
# Configuration
# -------------------------------
EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="key-vault-scope",
    key="eventhub-connection-string"
)
EVENT_HUB_NAME = "pmb-weather-streaming-eventhub"

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

# -------------------------------
# Event Hub
# -------------------------------
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    print(f"Event sent: {event.get('name')} @ {event.get('localtime')}")

# -------------------------------
# API Calls
# -------------------------------
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"{response.status_code}: {response.text}"}

def get_current_weather(base_url, api_key, location):
    response = requests.get(f"{base_url}/current.json", params={"key": api_key, "q": location, "aqi": "yes"})
    return handle_response(response)

def get_forecast_weather(base_url, api_key, location, days):
    response = requests.get(f"{base_url}/forecast.json", params={"key": api_key, "q": location, "days": days})
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    response = requests.get(f"{base_url}/alerts.json", params={"key": api_key, "q": location, "alerts": "yes"})
    return handle_response(response)

# -------------------------------
# Flatten Data
# -------------------------------
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast_days = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    return {
        "name": location_data.get("name"),
        "region": location_data.get("region"),
        "country": location_data.get("country"),
        "lat": location_data.get("lat"),
        "lon": location_data.get("lon"),
        "localtime": location_data.get("localtime"),
        "temp_c": current.get("temp_c"),
        "is_day": current.get("is_day"),
        "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"),
        "wind_kph": current.get("wind_kph"),
        "wind_degree": current.get("wind_degree"),
        "air_quality": {
            "co": air_quality.get("co"),
            "no2": air_quality.get("no2"),
            "o3": air_quality.get("o3"),
            "so2": air_quality.get("so2"),
            "pm2_5": air_quality.get("pm2_5"),
            "pm10": air_quality.get("pm10"),
            "us_epa_index": air_quality.get("us-epa-index"),
            "gb_defra_index": air_quality.get("gb-defra-index")
        },
        "alerts": [
            {
                "headline": alert.get("headline"),
                "severity": alert.get("severity"),
                "description": alert.get("desc"),
                "instruction": alert.get("instruction")
            }
            for alert in alert_list
        ],
        "forecast": [
            {
                "date": day.get("date"),
                "maxtemp_c": day.get("day", {}).get("maxtemp_c"),
                "mintemp_c": day.get("day", {}).get("mintemp_c"),
                "condition": day.get("day", {}).get("condition", {}).get("text")
            }
            for day in forecast_days
        ]
    }

# -------------------------------
# Main Fetch Function
# -------------------------------
def fetch_and_send_weather():
    base_url = "http://api.weatherapi.com/v1"
    location = "Chennai"
    api_key = dbutils.secrets.get(scope="key-vault-scope", key="apikey")

    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    alerts = get_alerts(base_url, api_key, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    send_event(merged_data)
    return merged_data

# -------------------------------
# Run Loop (simulates streaming)
# -------------------------------
try:
    while True:
        fetch_and_send_weather()
        time.sleep(60)  # Fetch every 60 seconds
except KeyboardInterrupt:
    print("Stopped.")
finally:
    producer.close()
    print("Producer closed.")




Event sent: Chennai @ 2026-03-28 10:56
Event sent: Chennai @ 2026-03-28 10:56
Event sent: Chennai @ 2026-03-28 10:56
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:59
Event sent: Chennai @ 2026-03-28 10:56


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

## Sending the Weather Data to Event Hub in every 30 Seconds


In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json
import requests
from datetime import datetime
import os

# -------------------------------
# Configuration
# -------------------------------
EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
    scope="key-vault-scope",
    key="eventhub-connection-string"
)
EVENT_HUB_NAME = "pmb-weather-streaming-eventhub"

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

# -------------------------------
# Event Hub
# -------------------------------
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    print(f"Event sent: {event.get('name')} @ {event.get('localtime')}")

# -------------------------------
# API Calls
# -------------------------------
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"{response.status_code}: {response.text}"}

def get_current_weather(base_url, api_key, location):
    response = requests.get(f"{base_url}/current.json", params={"key": api_key, "q": location, "aqi": "yes"})
    return handle_response(response)

def get_forecast_weather(base_url, api_key, location, days):
    response = requests.get(f"{base_url}/forecast.json", params={"key": api_key, "q": location, "days": days})
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    response = requests.get(f"{base_url}/alerts.json", params={"key": api_key, "q": location, "alerts": "yes"})
    return handle_response(response)

# -------------------------------
# Flatten Data
# -------------------------------
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast_days = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    return {
        "name": location_data.get("name"),
        "region": location_data.get("region"),
        "country": location_data.get("country"),
        "lat": location_data.get("lat"),
        "lon": location_data.get("lon"),
        "localtime": location_data.get("localtime"),
        "temp_c": current.get("temp_c"),
        "is_day": current.get("is_day"),
        "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"),
        "wind_kph": current.get("wind_kph"),
        "wind_degree": current.get("wind_degree"),
        "air_quality": {
            "co": air_quality.get("co"),
            "no2": air_quality.get("no2"),
            "o3": air_quality.get("o3"),
            "so2": air_quality.get("so2"),
            "pm2_5": air_quality.get("pm2_5"),
            "pm10": air_quality.get("pm10"),
            "us_epa_index": air_quality.get("us-epa-index"),
            "gb_defra_index": air_quality.get("gb-defra-index")
        },
        "alerts": [
            {
                "headline": alert.get("headline"),
                "severity": alert.get("severity"),
                "description": alert.get("desc"),
                "instruction": alert.get("instruction")
            }
            for alert in alert_list
        ],
        "forecast": [
            {
                "date": day.get("date"),
                "maxtemp_c": day.get("day", {}).get("maxtemp_c"),
                "mintemp_c": day.get("day", {}).get("mintemp_c"),
                "condition": day.get("day", {}).get("condition", {}).get("text")
            }
            for day in forecast_days
        ]
    }

# -------------------------------
# Fetch Weather Data
# -------------------------------
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1"
    location = "Chennai"
    api_key = dbutils.secrets.get(scope="key-vault-scope", key="apikey")

    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    alerts = get_alerts(base_url, api_key, location)

    return flatten_data(current_weather, forecast_weather, alerts)

# -------------------------------
# Batch Processor
# -------------------------------
def process_batch(batch_df, batch_id):
    try:
        weather_data = fetch_weather_data()
        send_event(weather_data)
        print(f"Event sent at {datetime.now()}")
    except Exception as e:
        print(f"Error in batch {batch_id}: {str(e)}")
        raise e

# -------------------------------
# Spark Structured Streaming
# -------------------------------
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

checkpoint_path = "file:///tmp/checkpoints/rate_stream"
os.makedirs("/tmp/checkpoints/rate_stream", exist_ok=True)

try:
    query = streaming_df.writeStream \
        .foreachBatch(process_batch) \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(processingTime="30 seconds") \
        .start()

    query.awaitTermination()

finally:
    producer.close()
    print("Producer closed.")

Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:14:16.525735
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:14:31.095391
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:15:00.816394
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:15:30.692651
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:16:00.792839
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:16:30.786367
Event sent: Chennai @ 2026-03-28 16:36
Event sent at 2026-03-28 11:17:00.819704
Event sent: Chennai @ 2026-03-28 16:33
Event sent at 2026-03-28 11:17:31.620249
Event sent: Chennai @ 2026-03-28 16:33
Event sent at 2026-03-28 11:18:00.767629
Event sent: Chennai @ 2026-03-28 16:33
Event sent at 2026-03-28 11:18:30.614273
Event sent: Chennai @ 2026-03-28 16:33
Event sent at 2026-03-28 11:19:00.792690
Event sent: Chennai @ 2026-03-28 16:33
Event sent at 2026-03-28 11:19:30.792298
Event sent: Chennai @ 2026-03-28 16:33
E

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can